# Случайный лес с логарифмом цены 

In [1]:
import pandas as pd
import numpy as np

In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import joblib
import pickle

```
Создание колонки с логарифмом цены
```

In [3]:
df = pd.read_csv('data/drom_archive_cleaned_2018-2025.csv')

In [4]:
df['log_price'] = np.log(df['Цена'])

```
Преобразование категориальных переменных
```

In [5]:
categorical = ['Название машины', 'Тип двигателя', 'Коробка передач', 'Привод', 'Руль', 'Поколение', 'Рестайлинг',
               'Тип кузова', 'Метка', 'Город']
numerical = ['Год', 'Объем двигателя', 'Мощность', 'Пробег', 'Год объявления', 'Месяц объявления', 'Возраст авто']

In [6]:
preprocessor = ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical), ('num', 'passthrough', numerical)], remainder='drop')

In [7]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1,
        max_depth=20,
        verbose=2
    ))
])

```
Загрузка обучающей и тестовой выборок
```

In [8]:
X_train, X_test, y_train, y_test = joblib.load("data/data_split_log.pkl")

```
Обучение и сохранение модели
```

In [9]:
model.fit(X_train, y_train)
joblib.dump(model, "models/rf_log_model.pkl")

['models/rf_log_model.pkl']

```
Предсказание на тестовой выборке
```

In [10]:
y_pred_log = model.predict(X_test)

[Parallel(n_jobs=6)]: Done  50 out of  50 | elapsed:    0.5s finished


In [11]:
y_pred = np.exp(y_pred_log) # Обратное преобразование прогнозов
y_test_real = np.exp(y_test)

```
Оценка модели
```

In [12]:
rf_mse = mean_squared_error(y_test_real, y_pred)
rf_rmse = np.sqrt(rf_mse)
rf_mae = mean_absolute_error(y_test_real, y_pred)
rf_r2 = r2_score(y_test_real, y_pred)

```
Вывод результатов
```

In [13]:
pd.options.display.float_format = '{:_.2f}'.format
pd.DataFrame({
    'Метод оценки': ['Среднеквадратическая ошибка (MSE)', 'Среднеквадратическая ошибка (RMSE)', 'Средняя абсолютная ошибка (MAE)', 'Коэффицент детерминации (R^2)'],
    'Результаты': [rf_mse, rf_rmse, rf_mae, rf_r2]
})

,Метод оценки,Результаты
0,Среднеквадратическая ошибка (MSE),103_541_256_750.19
1,Среднеквадратическая ошибка (RMSE),321_778.27
2,Средняя абсолютная ошибка (MAE),197_389.88
3,Коэффицент детерминации (R^2),0.90


```
Сохранение метрик в отдельный файл
```

In [14]:
metrics = {
    "model_name": "Random Forest Log",
    "MSE": rf_mse,
    "RMSE": rf_rmse,
    "MAE": rf_mae,
    "R2": rf_r2
}

In [15]:
with open("metrics/rf_log_metrics.pkl", "wb") as f:
    pickle.dump(metrics, f)